# 생성형 합성(gen) 혼합비 ablation — 가산 · 3시드

**사전 등록**: `docs/PREREGISTER_GEN_ABLATION.md` 를 먼저 읽을 것(결정 규칙·한계 확정본).

**무엇을 하나**: 누수 통제된 실 화재 학습셋에 gen 합성을 **가산**으로 섞어
`synth-only / real-only / 1:1 / 1:3` 4 arm 을 각 3시드 학습하고,
배포 대표 hold-out `oilfire_realtest` 에서 **장면단위 recall + fpr_급식실** 을 conf sweep 로 비교한다.

**안전**: 모든 재구성은 세션-로컬(`/content/...`). Drive 는 zip/데이터 **읽기만**,
모델은 **신규 네임스페이스 `runs_genabl`** 에만 기록(기존 `real_only_grouped` 등 미덮음).
`split_audit` 을 직접 돌리지 않는 이유 = 그 스크립트가 Drive `real_only_grouped`(2g)를 덮어쓰기 때문.

> VSCode 의 Colab/Jupyter 커널에서 위→아래로 실행. 처음엔 GPU 런타임 확인.

In [ ]:
# ── 셋업: Drive 마운트 · repo clone · 패키지 · 사전 커밋 상수 ─────────────
import os, sys, glob, json, shutil, subprocess, random, zipfile
FIRE = '/content/drive/MyDrive/fire_frames'
REPO = '/content/kitchen-fire-noise-poc'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('drive.mount 건너뜀:', e)

if not os.path.isdir(REPO):
    subprocess.run(['git','clone','-q','https://github.com/K-H-MOON/kitchen-fire-noise-poc', REPO], check=False)
else:
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
sys.path.insert(0, REPO + '/scripts')

for pkg in ('ultralytics','imagehash'):
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True)

import numpy as np
from PIL import Image
import imagehash
from ultralytics import YOLO

# 사전 커밋 상수(사전 등록 문서 §6 과 일치)
SEEDS       = [0, 1, 2]
SAMPLE_SEED = 1234
RATIOS      = {'synthonly': None, 'realonly': 0, 'r1s1': 1, 'r1s3': 3}  # real:synth 가산배수
EPOCHS, IMGSZ, PATIENCE = 60, 640, 15
HAM_THRESH, SPLIT_SEED  = 6, 0
CONF_GRID = [round(float(c),2) for c in np.arange(0.05, 0.86, 0.05)]
REF_CONF  = 0.25

ZIP     = '/content/drive/MyDrive/Indoor Fire Smoke.zip'
RAW     = '/content/Indoor_Fire_Smoke'
GROUPED = '/content/if_fire_grouped'
GEN_OUT = '/content/gen_synth'
TEST    = FIRE + '/oilfire_realtest'
RUNS    = FIRE + '/runs_genabl'                       # 신규 네임스페이스(기존 미덮음)
RESULT_JSON = FIRE + '/indoorfire_eval/gen_ablation_result.json'
os.makedirs(os.path.dirname(RESULT_JSON), exist_ok=True)
print('셋업 완료 · FIRE =', FIRE)

## 1) 합성 pool 빌드 — `colab_gen_build.py` 재사용 (`TRAIN=0`, Drive 미변경)

`GEN_SRC` 기본 = `{FIRE}/gen_export`(Roboflow YOLO export). 경로 다르면 아래 주석 해제.

In [ ]:
os.environ['GEN_OUT'] = GEN_OUT
os.environ['TRAIN']   = '0'            # 데이터셋만 — gen 앵커 학습/Drive 기록 안 함
# os.environ['GEN_SRC'] = FIRE + '/gen_export'
%run -i /content/kitchen-fire-noise-poc/scripts/colab_gen_build.py

# 합성 pool = gen_synth 의 라벨(박스) 있는 이미지 전부(내부 train/val 분할 무시)
gen_pool = []
for s in ('train','val'):
    for ip in sorted(glob.glob(GEN_OUT + '/' + s + '/images/*')):
        lp = GEN_OUT + '/' + s + '/labels/' + os.path.splitext(os.path.basename(ip))[0] + '.txt'
        if os.path.exists(lp) and os.path.getsize(lp) > 0:
            gen_pool.append((ip, lp))
G = len(gen_pool)
assert G, 'gen pool 0장 — GEN_SRC(Roboflow export) 확인'
print('합성 pool G =', G)

## 2) 실 pool 빌드 — 누수 통제 그룹 split (`split_audit` 로직 포팅)

dHash(Hamming≤6) 근접중복 → 그룹단위 70/15/15 재분할. 세션-로컬, **Drive 미변경**(zip 읽기만).
`split_audit` 과 동일 시드·임계라 검증된 split 을 그대로 재현한다.

In [ ]:
if not os.path.isdir(RAW) or not os.listdir(RAW):
    os.makedirs(RAW, exist_ok=True); print('Indoor 압축 해제...')
    zipfile.ZipFile(ZIP).extractall(RAW)

def is_fire(lp):
    if not (os.path.exists(lp) and os.path.getsize(lp) > 0): return False
    return any(l.split() and l.split()[0]=='0' for l in open(lp))

recs = []
for sp in ('train','valid','test'):
    for p in glob.glob(RAW + '/**/' + sp + '/images/*.jpg', recursive=True):
        lp = p.replace(os.sep+'images'+os.sep, os.sep+'labels'+os.sep)[:-4] + '.txt'
        recs.append(dict(img=p, split=sp, label=lp, fire=is_fire(lp)))
N = len(recs); assert N, 'Indoor 이미지 0장 — ' + ZIP + ' 확인'
print('총', N, '장')

print('dHash 계산...')
def phash_u64(path):
    h = imagehash.dhash(Image.open(path).convert('RGB')); v = 0
    for b in h.hash.flatten(): v = (v<<1) | int(b)
    return np.uint64(v)
hashes = np.empty(N, dtype=np.uint64)
for i,r in enumerate(recs):
    try: hashes[i] = phash_u64(r['img'])
    except Exception: hashes[i] = np.uint64(0)

POP = np.array([bin(i).count('1') for i in range(1<<16)], dtype=np.uint8)
def hamm_to_all(h):
    x = hashes ^ h
    return (POP[np.asarray(x & np.uint64(0xFFFF), dtype=np.uint32)]
          + POP[np.asarray((x>>np.uint64(16)) & np.uint64(0xFFFF), dtype=np.uint32)]
          + POP[np.asarray((x>>np.uint64(32)) & np.uint64(0xFFFF), dtype=np.uint32)]
          + POP[np.asarray((x>>np.uint64(48)) & np.uint64(0xFFFF), dtype=np.uint32)])
parent = list(range(N))
def find(a):
    while parent[a]!=a: parent[a]=parent[parent[a]]; a=parent[a]
    return a
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[max(ra,rb)]=min(ra,rb)
print('클러스터링 Hamming <=', HAM_THRESH, '...')
for i in range(N):
    d = hamm_to_all(hashes[i])
    for j in np.where(d<=HAM_THRESH)[0]:
        if j>i: union(i,int(j))
gid = np.array([find(i) for i in range(N)])
groups = {}
for i,g in enumerate(gid): groups.setdefault(int(g),[]).append(i)

fire_of = np.array([r['fire'] for r in recs])
rng = np.random.default_rng(SPLIT_SEED)
gids = list(groups.keys()); rng.shuffle(gids)
tr_cap, va_cap = int(N*0.70), int(N*0.15)
assign={}; c_tr=c_va=0
for g in gids:
    n=len(groups[g])
    if c_tr+n<=tr_cap or c_tr==0: assign[g]='train'; c_tr+=n
    elif c_va+n<=va_cap or c_va==0: assign[g]='valid'; c_va+=n
    else: assign[g]='test'
new_split = np.array([assign[int(gid[i])] for i in range(N)])

if os.path.isdir(GROUPED): shutil.rmtree(GROUPED)
for i,r in enumerate(recs):
    sp=new_split[i]; di=GROUPED+'/'+sp+'/images'; dl=GROUPED+'/'+sp+'/labels'
    os.makedirs(di,exist_ok=True); os.makedirs(dl,exist_ok=True)
    name=os.path.basename(r['img']); stem=os.path.splitext(name)[0]; dst=di+'/'+name
    if not os.path.exists(dst): os.symlink(os.path.realpath(r['img']), dst)
    lines=[l for l in open(r['label']) if l.split() and l.split()[0]=='0'] if os.path.exists(r['label']) else []
    open(dl+'/'+stem+'.txt','w').writelines(lines)
open(GROUPED+'/data.yaml','w').write(
    "path: "+GROUPED+"\ntrain: train/images\nval: valid/images\ntest: test/images\nnc: 1\nnames: ['fire']\n")
for sp in ('train','valid','test'):
    m=new_split==sp; print('  '+sp+':', int(m.sum()), '장 (fire', int((m&fire_of).sum()), ')')
real_train = sorted(glob.glob(GROUPED + '/train/images/*'))
R = len(real_train); print('실 train R =', R)

## 3) arm 데이터셋 빌드 — 가산, 표본 고정

`val` 은 전 arm 공유(= 실 `valid`) → 부차 box mAP 를 arm 간 동일 셋에서 비교. gen 부표본은
`SAMPLE_SEED` 로 한 번 뽑아 3시드에 동일 재사용(데이터 변동원 제거). `3R > G` 면 복원추출(중복 로그).

In [ ]:
VAL_IMAGES = GROUPED + '/valid/images'
sampler = random.Random(SAMPLE_SEED)

def link_img(src, dst):
    if not os.path.exists(dst):
        try: os.symlink(os.path.realpath(src), dst)
        except Exception: shutil.copy(src, dst)

def build_arm(name, ratio):
    root='/content/arm_'+name
    if os.path.isdir(root): shutil.rmtree(root)
    di=root+'/images'; dl=root+'/labels'; os.makedirs(di,exist_ok=True); os.makedirs(dl,exist_ok=True)
    n_real=n_syn=0
    if ratio is not None:                       # real 포함(real-only, r1s1, r1s3)
        for ip in real_train:
            nm=os.path.basename(ip); stem=os.path.splitext(nm)[0]
            link_img(ip, di+'/'+nm)
            sl=GROUPED+'/train/labels/'+stem+'.txt'
            open(dl+'/'+stem+'.txt','w').writelines(open(sl).readlines() if os.path.exists(sl) else [])
            n_real+=1
    if ratio is None:      picks=list(gen_pool)
    elif ratio==0:         picks=[]
    else:
        need=ratio*R
        picks = sampler.sample(gen_pool, need) if need<=G else [gen_pool[sampler.randrange(G)] for _ in range(need)]
    for k,(ip,lp) in enumerate(picks):
        stem='gen%05d_'%k + os.path.splitext(os.path.basename(ip))[0]; ext=os.path.splitext(ip)[1]
        link_img(ip, di+'/'+stem+ext)
        open(dl+'/'+stem+'.txt','w').writelines(open(lp).readlines()); n_syn+=1
    n_dup = (ratio*R - G) if (ratio not in (None,0) and ratio*R>G) else 0
    open(root+'/data.yaml','w').write(
        "path: "+root+"\ntrain: images\nval: "+VAL_IMAGES+"\nnc: 1\nnames: ['fire']\n")
    print('[%-9s] real %d · synth %d%s · 총 %d' % (name, n_real, n_syn,
          (' (중복 %d)'%n_dup if n_dup else ''), n_real+n_syn))
    return root+'/data.yaml'

ARMS = {name: build_arm(name, r) for name,r in RATIOS.items()}

## 4) 학습 — 4 arm × 3 시드 = 12

`runs_genabl/<arm>_s<seed>`(Drive 신규). 부차 지표로 공유 실 valid box mAP 도 함께 기록.

In [ ]:
trained, val_map = {}, {}
for arm, yml in ARMS.items():
    for s in SEEDS:
        run = arm + '_s' + str(s); print('\n===== 학습', run, '=====')
        YOLO('yolov8s.pt').train(data=yml, epochs=EPOCHS, imgsz=IMGSZ, patience=PATIENCE,
             seed=s, deterministic=True, project=RUNS, name=run, exist_ok=True,
             verbose=False, plots=False)
        best = RUNS + '/' + run + '/weights/best.pt'; trained[(arm,s)] = best
        try:
            mv = YOLO(best).val(data=yml, split='val', verbose=False)
            val_map[(arm,s)] = (float(mv.box.map50), float(mv.box.map))
        except Exception as e:
            val_map[(arm,s)] = (float('nan'), float('nan')); print('val 실패:', e)
print('\n학습 완료:', len(trained), '모델')

## 5) hold-out 평가 — frame-level recall/fpr, conf sweep

이미지별 max box conf 를 한 번 뽑고 임계를 numpy 로 쓸어봄(모델당 1 predict pass).
recall 은 **장면단위** 평균(N=장면수, std 동반).

In [ ]:
def scene(p): return os.path.basename(p).rsplit('_',1)[0]
fire_imgs = sorted(glob.glob(TEST + '/fire/*.jpg'))
cook_imgs = sorted(glob.glob(TEST + '/nofire_kitchen/*.jpg'))
pre_imgs  = sorted(glob.glob(TEST + '/nofire_presrc/*.jpg'))
assert fire_imgs, '양성 없음: ' + TEST + '/fire'
fire_scene = np.array([scene(p) for p in fire_imgs]); scenes = sorted(set(fire_scene))
print('test: fire', len(fire_imgs), '· 급식실', len(cook_imgs), '· 발화전', len(pre_imgs))
print('장면:', scenes)

def max_confs(model, paths):
    out=[]
    for p in paths:
        r=model.predict(p, conf=0.01, imgsz=IMGSZ, verbose=False)[0]
        out.append(float(r.boxes.conf.max()) if len(r.boxes) else 0.0)
    return np.array(out)

def sweep(mcf, mck, mcp):
    rows=[]
    for c in CONF_GRID:
        df=(mcf>=c).astype(float)
        ps={sc: float(df[fire_scene==sc].mean()) for sc in scenes}
        rows.append(dict(conf=c, recall=float(np.mean(list(ps.values()))),
            recall_std=float(np.std(list(ps.values()))),
            fpr_kitchen=float((mck>=c).mean()) if len(mck) else float('nan'),
            fpr_presrc=float((mcp>=c).mean()) if len(mcp) else float('nan'),
            per_scene=ps))
    return rows

curves={}
for (arm,s),best in trained.items():
    m=YOLO(best)
    curves[(arm,s)]=sweep(max_confs(m,fire_imgs), max_confs(m,cook_imgs), max_confs(m,pre_imgs))
    print('  평가 완료', arm, 's'+str(s))

## 6) 판정 — 매칭 recall / 매칭 fpr (사전 등록 §3)

비교는 고정 conf 가 아니라 **매칭 운영점**에서. 기준 = real-only 3시드 평균의 `conf 0.25` 운영점.

In [ ]:
def at_conf(rows, c): return min(rows, key=lambda r: abs(r['conf']-c))
def interp_fpr_at_recall(rows, tgt):
    xs=np.array([r['recall'] for r in rows]); ys=np.array([r['fpr_kitchen'] for r in rows])
    o=np.argsort(xs); return float(np.interp(tgt, xs[o], ys[o]))
def interp_recall_at_fpr(rows, tgt):
    xs=np.array([r['fpr_kitchen'] for r in rows]); ys=np.array([r['recall'] for r in rows])
    o=np.argsort(xs); return float(np.interp(tgt, xs[o], ys[o]))
def agg(v): return (float(np.mean(v)), float(np.std(v)))

ro=[at_conf(curves[('realonly',s)], REF_CONF) for s in SEEDS]
recall_star=float(np.mean([r['recall'] for r in ro]))
fpr_star   =float(np.mean([r['fpr_kitchen'] for r in ro]))
print('기준(real-only @conf%.2f): recall*=%.3f · fpr_급식실*=%.3f\n' % (REF_CONF, recall_star, fpr_star))

hdr='%-10s %12s %10s | %12s %12s %9s' % ('arm','recall@ref','fpr@ref','fpr@recall*','recall@fpr*','valmAP50')
print(hdr); summary={}
for arm in RATIOS:
    ref=[at_conf(curves[(arm,s)], REF_CONF) for s in SEEDS]
    rec_ref=agg([r['recall'] for r in ref]); fpr_ref=agg([r['fpr_kitchen'] for r in ref])
    fpr_m =agg([interp_fpr_at_recall(curves[(arm,s)], recall_star) for s in SEEDS])
    rec_m =agg([interp_recall_at_fpr(curves[(arm,s)], fpr_star) for s in SEEDS])
    vm    =agg([val_map[(arm,s)][0] for s in SEEDS])
    summary[arm]=dict(recall_ref=rec_ref, fpr_ref=fpr_ref, fpr_at_recallstar=fpr_m,
                      recall_at_fprstar=rec_m, val_map50=vm)
    print('%-10s %.3f±%.3f %.3f±%.3f | %.3f±%.3f %.3f±%.3f %.3f' % (arm,
          rec_ref[0],rec_ref[1], fpr_ref[0],fpr_ref[1], fpr_m[0],fpr_m[1], rec_m[0],rec_m[1], vm[0]))

base=summary['realonly']['fpr_at_recallstar']
print('\n── 판정(사전 등록 §3, 가산 arm vs real-only) ──')
for arm in ('r1s1','r1s3'):
    d=summary[arm]['fpr_at_recallstar'][0]-base[0]
    pooled=(summary[arm]['fpr_at_recallstar'][1]+base[1]) or 1e-9
    v='도움됨(확정후보)' if d<-pooled else ('해로움(후보)' if d>pooled else '무기여(시드노이즈 내)')
    print('  %s: 매칭recall fpr_급식실 Δ=%+.3f (arm std %.3f, base std %.3f) → %s' % (
          arm, d, summary[arm]['fpr_at_recallstar'][1], base[1], v))
print('\n※ synth-only recall@ref = %.3f (gen 단독 전이 맥락) · sc14 아래 셀 별도(약한 신호).' % summary['synthonly']['recall_ref'][0])

json.dump({'recall_star':recall_star,'fpr_star':fpr_star,'ref_conf':REF_CONF,'seeds':SEEDS,
           'ratios':RATIOS,'R':R,'G':G,'summary':summary,
           'curves':{a+'_s'+str(s):curves[(a,s)] for (a,s) in curves}},
          open(RESULT_JSON,'w'), ensure_ascii=False, indent=1)
print('\n-> 저장:', RESULT_JSON)

## 7) 그림 + 최난 장면(sc14) 별도

recall–fpr 곡선(arm 평균)과, 역대 어려운 장면의 arm별 recall(**N 작음 = 탐색적 신호, 확정 근거 아님**).
`HARD` 는 위 셀이 출력한 `장면:` 목록을 보고 맞춰 조정.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.figure(figsize=(6,5))
for arm in RATIOS:
    rs=[np.mean([curves[(arm,s)][i]['recall']      for s in SEEDS]) for i in range(len(CONF_GRID))]
    fs=[np.mean([curves[(arm,s)][i]['fpr_kitchen'] for s in SEEDS]) for i in range(len(CONF_GRID))]
    plt.plot(fs, rs, marker='.', label=arm)
plt.xlabel('fpr_급식실'); plt.ylabel('recall(장면단위)'); plt.grid(alpha=.3); plt.legend()
plt.title('recall vs fpr_급식실 (arm 평균, conf sweep)'); plt.show()

HARD = ['sc14']       # 필요시 위 '장면:' 목록 보고 조정
for hs in HARD:
    print('\n[%s] arm별 recall@conf%.2f (평균±std, N작음=탐색적):' % (hs, REF_CONF))
    for arm in RATIOS:
        vals=[min(curves[(arm,s)], key=lambda r: abs(r['conf']-REF_CONF))['per_scene'].get(hs, float('nan')) for s in SEEDS]
        print('  %-10s %.3f±%.3f' % (arm, np.nanmean(vals), np.nanstd(vals)))